In [1]:
import numpy as np
import pandas as pd 
import glob 

In [2]:
files = sorted(glob.glob("ENERGY_*.csv"))

df_energy = None
for file in files:
    temp = pd.read_csv(file)
    temp["entity_id"] = (
        temp["entity_id"].str.replace("sensor.", "", regex=False)
        .str.replace("_month_energy", "", regex=False)
    )
    temp = temp.drop(columns=["type", "unit"], errors="ignore")

    # Fixed: Changed 'df' to 'df_energy'
    if df_energy is None:
        df_energy = temp
    else:
        df_energy = df_energy.merge(temp, on="entity_id", how="outer")

# Transpose so timestamps are rows, entity_ids are columns
df_energy = df_energy.set_index("entity_id").T

# Fixed: Ensure we are renaming and converting the index of df_energy
df_energy.index.name = "timestamp"
df_energy.index = pd.to_datetime(df_energy.index)

In [6]:
df_occupancy = pd.read_csv(
    "../OccupantData/OccupancyLog_05202026.csv",
    parse_dates=["timestamp_start", "timestamp_end"]
)

In [13]:
df_energy.head()

entity_id,brannigan,bruce_s_desk,coffee_machine,crux_monitor,demo_monitors,dishwasher,hallway_monitor,main_fridge,heating_water,meeting_room_monitor_right,meeting_room_monitor_1,meeting_room_monitor_2,microwave_bottom,milk_fridge,printer
timestamp,,,,,,,,,,,,,,,
2026-05-19 16:00:00+00:00,0.0,NaN,0.003,NaN,0.075,NaN,0.035,0.136,NaN,0.035,0.048,0.007,0.0,0.026,0.028
2026-05-19 16:05:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-05-19 16:10:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-05-19 16:15:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-05-19 16:20:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
# Forward fill the constant power state
df_energy_filled = df_energy.ffill().fillna(0)

In [7]:
df_occupancy.head()

,timestamp_start,timestamp_end,entrances,avg_occupancy,temp_c,humidity_pct
0,2026-05-19 08:47:06.032000+00:00,2026-05-19 08:48:06.032000+00:00,0,0.050,22.0,47
1,2026-05-19 08:48:06.098000+00:00,2026-05-19 08:49:06.098000+00:00,0,0.289,22.0,47
2,2026-05-19 08:49:06.262000+00:00,2026-05-19 08:50:06.262000+00:00,2,1.199,22.0,47
3,2026-05-19 08:50:06.426000+00:00,2026-05-19 08:51:06.426000+00:00,0,0.847,22.0,47
4,2026-05-19 08:51:06.490000+00:00,2026-05-19 08:52:06.490000+00:00,1,0.664,22.0,47


In [8]:
# 1. Convert occupancy timestamps to datetime objects
df_occupancy["timestamp_start"] = pd.to_datetime(
    df_occupancy["timestamp_start"]
)

# 2. Sort both datasets by their timestamps (REQUIRED for merge_asof)
df_occupancy = df_occupancy.sort_values("timestamp_start")
df_energy = df_energy.sort_index()

# 3. Reset the energy index so 'timestamp' becomes a normal column for the merge
df_energy_col = df_energy.reset_index()

# 4. Perform the smart merge
# This matches each energy timestamp to the CLOSEST occupancy record before it
df_combined = pd.merge_asof(
    df_energy_col,
    df_occupancy,
    left_on="timestamp",
    right_on="timestamp_start",
    direction="backward",  # Finds the most recent occupancy record
    tolerance=pd.Timedelta("5min"),  # Prevents matching if data is missing
)

# 5. Set timestamp back as the index
df_combined = df_combined.set_index("timestamp")

In [11]:
df_occupied_only = df_combined[df_combined["avg_occupancy"].notna()]

# Display the first few rows
df_occupied_only.head()

,brannigan,bruce_s_desk,coffee_machine,crux_monitor,demo_monitors,dishwasher,hallway_monitor,main_fridge,heating_water,meeting_room_monitor_right,...,meeting_room_monitor_2,microwave_bottom,milk_fridge,printer,timestamp_start,timestamp_end,entrances,avg_occupancy,temp_c,humidity_pct
timestamp,,,,,,,,,,,,,,,,,,,,,
2026-05-19 19:15:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2026-05-19 19:12:26.660000+00:00,2026-05-19 19:13:26.660000+00:00,0.0,0.060,22.0,47.0
2026-05-20 00:00:00+00:00,0.001,NaN,0.171,NaN,0.077,NaN,0.085,0.079,NaN,0.097,...,0.007,0.001,0.026,0.028,2026-05-19 23:59:03.694000+00:00,2026-05-20 00:00:03.694000+00:00,2.0,0.140,22.0,47.0
2026-05-20 02:05:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2026-05-20 02:04:19.851000+00:00,2026-05-20 02:05:19.851000+00:00,2.0,0.233,22.0,47.0
2026-05-20 02:10:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2026-05-20 02:09:20.473000+00:00,2026-05-20 02:10:20.473000+00:00,4.0,0.312,22.0,47.0
2026-05-20 03:10:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2026-05-20 03:08:28.273000+00:00,2026-05-20 03:09:28.273000+00:00,1.0,0.053,22.0,47.0
